In [107]:
from spin_lattices import KagomeLattice, SpinLattice, ChainLattice, SquareLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from pathlib import Path
import networkx as nx
import numpy as np
from typing import Callable
import torch
import numpy.typing as npt
import lattice_symmetries as ls
from typing import Any, Optional, Union, Dict, Tuple
from loguru import logger
from collections import namedtuple
from torch import Tensor
import torch.nn as nn
from misc_utils import make_unpacked_configurations
import io
from contextlib import redirect_stderr
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from nqs_playground_helpers import SamplingOptions, split_into_batches, safe_exp, sample_exactly
from scipy.sparse import csr_matrix, coo_matrix, diags
from scipy.sparse.csgraph import connected_components

In [2]:
def find_overlap(x, y):
    x = x.view(-1)
    y = y.view(-1)
    return torch.sum(x * y) / torch.sqrt(torch.sum(x**2) * torch.sum(y**2))

In [3]:
# lattice = ChainLattice(10)
# system = HeisenbergJ1J2(
#     lattice=lattice,
#     J1=1,
#     J2=1,
#     ground_state_cache_dir=Path("groundstates"),
#     use_symmetries=False,
#     spin_inversion=None,
# )
# system.hamiltonian.apply_off_diag_to_basis_state(system.hamiltonian.basis.states[0])

2023-07-26 19:38:24.407 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=10
2023-07-26 19:38:24.408 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 0 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-26 19:38:24.417 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 252


[((2+0j), 542), ((2+0j), 47)]

In [2]:
class LogProbDenseNet(nn.Module):
    def __init__(self, system: SpinSystem, n_hidden: int = 100):
        super().__init__()
        self.system = system
        self.n_hidden = n_hidden
        self.net = nn.Sequential(
            nn.Linear(system.number_spins, n_hidden), nn.ReLU(), nn.Linear(n_hidden, 1)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(
            torch.from_numpy(
                make_unpacked_configurations(x, self.system.number_spins).astype(np.float32)
            )
        )

In [8]:
def find_nbd(
    system: SpinSystem, states: npt.NDArray[np.uint64]
) -> tuple[csr_matrix, npt.NDArray[np.uint64]]:
    """
    Constructs a sparse matrix that is a slice of the Hamiltonian matrix.


    Parameters
    ----------
    system : SpinSystem
        The system to construct the matrix for.

    states : npt.NDArray[np.uint64]
        The states whose neighbors to include in the matrix.

    Returns
    -------
    M : csr_matrix
        The sparse matrix.
    
    nbd_states : npt.NDArray[np.uint64]
        The sorted array of neighbors.
        
        The following holds:

        ``M[i, j] = <nbd_states[i] | H | nbd_states[j]>``
    """
    coeff_rows = []
    nbd_states_rows = []
    row_indices = [0]
    for state in states:
        # process neighbors
        cur_coeffs, cur_nbd_states = map(
            list, zip(*system.hamiltonian.apply_off_diag_to_basis_state(state))
        )

        # process self
        cur_coeffs.append(system.hamiltonian.apply_diag_to_basis_state(state))
        cur_nbd_states.append(state)

        # make rows
        coeff_rows.append(cur_coeffs)
        nbd_states_rows.append(cur_nbd_states)
        row_indices.append(row_indices[-1] + len(cur_coeffs))

    coeffs_data = np.concatenate(coeff_rows)
    nbd_states_data = np.concatenate(nbd_states_rows)
    nbd_states = np.unique(nbd_states_data)
    nbd_indices = np.searchsorted(nbd_states, nbd_states_data)
    row_indices = np.array(row_indices)

    return (
        csr_matrix((coeffs_data, nbd_indices, row_indices), shape=(len(states), len(nbd_states))),
        nbd_states,
    )

In [70]:
def nbd_matrix_to_graph(
    states: npt.NDArray, nbd_matrix: csr_matrix, nbd_states: npt.NDArray
) -> csr_matrix:
    """
    Constructs a graph from a neighborhood matrix (see ``find_nbd``):

    - Expands matrix to make it square. Rows are rearranged to align them
        with columns, indexed by ``nbd_states``. I.e. row ``i`` corresponds to
        ``nbd_states[i]``.

    - Symmetrizes the matrix.

    - Converts to a graph by thresholding at 0.
    """
    symmetric_matrix = csr_matrix((len(nbd_states), len(nbd_states)))
    state_indices = np.searchsorted(nbd_states, states)
    symmetric_matrix[state_indices, :] = (nbd_matrix != 0).astype(np.uint8)
    symmetric_matrix += symmetric_matrix.T
    return (symmetric_matrix != 0).astype(np.uint8)

In [90]:
def true_relsigns(system: SpinSystem) -> Callable[[npt.NDArray], npt.NDArray]:
    def relings(cluster):
        return np.sign(system.get_ground_state_coeffs(cluster)) * np.random.choice([-1, 1])

    return relings

In [108]:
def move_signs_to_H(states: npt.NDArray, M: csr_matrix, nbd_states: npt.NDArray, relsign_fn):
    """
    Moves the signs from the relative signs to the Hamiltonian matrix.
    """
    graph = nbd_matrix_to_graph(states, M, nbd_states)
    _, labels = connected_components(graph, directed=False)
    relsigns = np.empty(len(nbd_states), dtype=np.int8)
    for component in np.unique(labels):
        component_indices = np.where(labels == component)[0]
        cluster = nbd_states[component_indices]
        relsigns[component_indices] = relsign_fn(cluster)
        
    state_indices = np.searchsorted(nbd_states, states)

    return diags(relsigns[state_indices]) @ M @ diags(relsigns)

In [112]:
states = system.hamiltonian.basis.states[7:10]
M, nbd_states = find_nbd(system, states)
M = move_signs_to_H(states, M, nbd_states, true_relsigns(system))
M.todense()

/nix/store/810mm036773a280pdrx1k2pqh5pi1mmi-python3-3.10.12-env/lib/python3.10/site-packages/scipy/sparse/_index.py:137: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray_sparse(i, j, x)


matrix([[-2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j, -2.+0.j, -2.+0.j,  0.+0.j,
          0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j,
         -2.+0.j,  0.+0.j,  0.+0.j],
        [ 0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j, -2.+0.j, -2.+0.j,
          0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,
          0.+0.j, -2.+0.j,  0.+0.j],
        [ 0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j, -2.+0.j,
         -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j,  0.+0.j,  0.+0.j, -2.+0.j,
          0.+0.j,  0.+0.j, -2.+0.j]])

In [ ]:
n_samples = 1024
lr = 1e-3
batch_size = 64

lattice = ChainLattice(10)
system = HeisenbergJ1J2(
    lattice=lattice,
    J1=1,
    J2=1,
    ground_state_cache_dir=Path("groundstates"),
    use_symmetries=False,
    spin_inversion=None,
)
system.get_eigenstates(1)
eval_set = system.canonical_basis.states

log_prob_fn = LogProbDenseNet(system, n_hidden=32)
optimizer = torch.optim.Adam(log_prob_fn.parameters(), lr=lr)

true_amplitudes = torch.from_numpy(np.abs(system.get_ground_state_coeffs(eval_set)))

writer = SummaryWriter(
    log_dir=(
        f"experiments/2023_07_10/{datetime.now().strftime('%H_%M_%S')}"
    )
)
for step in range(100):
    states, _ = sample_exactly(
        log_prob_fn,
        system.basis,
        SamplingOptions(number_samples=n_samples, number_chains=1, mode="exact"),
    )
    states, weights = torch.unique(states.view(-1), return_counts=True)
    weights = weights.float() / torch.sum(weights)

    submatrix, cur_nbd_states = find_nbd(system, states.detach().numpy())
    E = find_local_energies(system, submatrix, states, cur_nbd_states, log_prob_fn)

    # states = states.view(-1, states.size(-1))
    # log_probs = log_probs.view(-1)
    # weights = weights.view(-1)

    # Compute output gradient
    with torch.no_grad():
        grad = E - E @ weights
        grad *= 4 * weights
        # coeff 4 is due to: 2 from formula, 2 due to we are working with log probs
        # instead of log amplitudes

        grad = grad.view(-1, 1)
        grad_norm = torch.linalg.norm(grad)
        logger.info("‖∇E‖₂ = {}", grad_norm)
        writer.add_scalar("loss/grad", grad_norm, step)

    optimizer.zero_grad()
    # batch_size = self.config.inference_batch_size

    # Computing gradients for the amplitude network
    logger.info("Computing gradients...")
    # if _should_optimize(self.config.amplitude):
    #     self.config.amplitude.train()
    forward_fn = log_prob_fn
    for states_chunk, grad_chunk in split_into_batches((states.view(-1, 1), grad), batch_size):
        output = forward_fn(states_chunk.view(-1))
        output.backward(grad_chunk)  # , retain_graph=True)
    
    optimizer.step()
    
    predicted_amplitudes = torch.exp(log_prob_fn(eval_set) * 0.5)
    
    overlap_ = find_overlap(true_amplitudes, predicted_amplitudes)
    writer.add_scalar("overlap", overlap_, step)